# 2. COMET Cell Classification Workflow

This is notebook 2 of 2 in the COMET workflow. It starts from the COMET/NIMBUS per-cell marker score tables generated by notebook 1 and produces thresholded marker calls, optional mutex-corrected positivity labels, ordered cell-type assignments, and `Additional_Labels` for secondary class matches.

If you want to reconstruct the final classified detections in QuPath, this notebook can optionally export `nimbus_cell_table_classified_qupath.csv`, which can then be used with `Qupath/Import COMET masks and cell class into Qupath.groovy`.

## Stage Map

1. **Locate NIMBUS cell tables** generated by `1_COMET_Workflow.ipynb`.
2. **Preview marker columns** from one slide and confirm the marker names that should be thresholded.
3. **Compute thresholds and review distributions** on a preview slide.
4. **Run batch classification** across all slides using optional `mutex_pairs` and ordered `cell_type_rules`.
5. **Optionally export** the classified tables as `fov`/`label`-keyed CSV files for downstream review or QuPath-side integration.

## Current Classification Logic

- `col_map=None` means marker names are matched directly against NIMBUS output column names.
- `factors` are optional; unspecified markers default to `1.0`.
- `mutex_pairs` is optional. For each configured pair, if both markers are positive in one cell, the weaker raw score is flipped to negative.
- `cell_type_rules` are evaluated **in the order you provide**. Put more specific classes before broader parent classes. The first matched rule becomes `Cell_Type`; any later matches are written to `Additional_Labels`.
- If a cell matches no class rule, `Cell_Type` remains `Unknown`.


## Expected Inputs And Outputs

### Input

```text
<BASE_DIR>/<slide>/nimbus_output/nimbus_cell_table.csv
```

### Outputs

```text
<BASE_DIR>/<slide>/nimbus_output/nimbus_cell_table_classified.csv
<BASE_DIR>/<slide>/nimbus_output/thresholds_used.csv
<BASE_DIR>/<slide>/nimbus_output/threshold_distributions.png
<BASE_DIR>/<slide>/nimbus_output/nimbus_cell_table_classified_qupath.csv  # optional
```

The optional CSV export is keyed by `fov` and `label`.


In [ ]:
# Cell strategy:
# Bootstrap the local COMET source tree into sys.path and import the
# current public classification/export APIs from the repository.

import sys
from pathlib import Path

import pandas as pd

CANDIDATE_ROOTS = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "COMET",
]

for candidate in CANDIDATE_ROOTS:
    if (candidate / "comet").exists():
        PROJECT_ROOT = candidate.resolve()
        break
else:
    raise FileNotFoundError(
        "Could not locate the COMET package directory. "
        "Open the notebook from the repository or add the package root to sys.path."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Using COMET source from: {PROJECT_ROOT}")

from comet import compute_thresholds, plot_marker_thresholds, threshold_slide, export_to_qupath


In [ ]:
# Cell strategy:
# Point the notebook at the experiment root and discover slide-level
# NIMBUS cell tables generated by the upstream workflow.

BASE_DIR = PROJECT_ROOT / "data" / "example"
NIMBUS_REL_PATH = Path("nimbus_output") / "nimbus_cell_table.csv"

slide_csvs = []
if BASE_DIR.exists():
    for slide_dir in sorted([d for d in BASE_DIR.iterdir() if d.is_dir()]):
        csv_path = slide_dir / NIMBUS_REL_PATH
        if csv_path.exists():
            slide_csvs.append(csv_path)

print(f"BASE_DIR: {BASE_DIR}")
print(f"Found {len(slide_csvs)} NIMBUS cell table(s).")
for csv_path in slide_csvs:
    print(f"  - {csv_path.relative_to(BASE_DIR)}")


In [ ]:
# Cell strategy:
# Load one preview slide so the user can inspect the available columns
# before deciding how markers, col_map, mutex_pairs, and class rules
# should be configured.

PREVIEW_SLIDE_INDEX = 0

if slide_csvs:
    preview_csv = slide_csvs[PREVIEW_SLIDE_INDEX]
    preview_table = pd.read_csv(preview_csv)
    print(f"Preview slide: {preview_csv}")
    print(f"Shape: {preview_table.shape}")
    print("Columns:")
    for col in preview_table.columns:
        print(f"  - {col}")
    display(preview_table.head())
else:
    print("No NIMBUS cell tables found. Please check BASE_DIR and upstream outputs.")


In [ ]:
# Cell strategy:
# Edit the classification configuration below to match the current
# experiment. The rule order matters: place more specific classes
# before broader parent classes; first match -> Cell_Type; later
# matches -> Additional_Labels.

MARKERS = [
    "CD3",
    "CD4",
    "CD8a",
    "TCRbeta",
    "EOMES",
]

FACTORS = {
    # Example: "EOMES": 1.3,
}

COL_MAP = None
# Example if your preferred marker name differs from the NIMBUS column:
# COL_MAP = {"TCR": "TCRbeta"}

MUTEX_PAIRS = [
    ("CD4", "CD8a"),
]

CELL_TYPE_RULES = [
    {"name": "CD4_TCell", "positive": ["CD3", "CD4"], "negative": ["CD8a"]},
    {"name": "CD8_TCell", "positive": ["CD3", "CD8a"], "negative": ["CD4"]},
    {"name": "TCell", "positive": ["CD3"], "negative": []},
    {"name": "EOMES_Pos", "positive": ["EOMES"], "negative": []},
]

MANUAL_THRESHOLDS = {
    # Example: "EOMES": 0.42,
}

PLOT_DISTRIBUTIONS = True
EXPORT_CLASSIFIED_CSV = True
EXPORT_CLASS_COLUMNS = ["Cell_Type"]


In [ ]:
# Cell strategy:
# Compute thresholds on the preview slide and visualize the marker
# distributions before launching the batch classification step.

if slide_csvs:
    preview_csv = slide_csvs[PREVIEW_SLIDE_INDEX]
    preview_table = pd.read_csv(preview_csv)
    threshold_info = compute_thresholds(
        cell_table=preview_table,
        markers=MARKERS,
        factors=FACTORS,
        col_map=COL_MAP,
    )

    threshold_df = pd.DataFrame.from_dict(threshold_info, orient="index")
    display(threshold_df)

    if threshold_info:
        plot_marker_thresholds(preview_table, threshold_info)
else:
    print("No preview slide available.")


In [ ]:
# Cell strategy:
# Run the full threshold/classification pipeline across every slide
# that contains a NIMBUS cell table. This writes classified CSV files,
# threshold summaries, and optional distribution plots per slide.

classification_outputs = []

for csv_path in slide_csvs:
    output_csv = csv_path.parent / f"{csv_path.stem}_classified.csv"

    threshold_slide(
        nimbus_csv=str(csv_path),
        markers=MARKERS,
        factors=FACTORS,
        col_map=COL_MAP,
        mutex_pairs=MUTEX_PAIRS,
        cell_type_rules=CELL_TYPE_RULES,
        manual_thresholds=MANUAL_THRESHOLDS,
        output_csv=str(output_csv),
        plot=PLOT_DISTRIBUTIONS,
    )

    classification_outputs.append({
        "slide": csv_path.parent.parent.name,
        "nimbus_csv": str(csv_path),
        "classified_csv": str(output_csv),
        "threshold_csv": str(output_csv.parent / "thresholds_used.csv"),
        "plot_path": str(output_csv.parent / "threshold_distributions.png"),
    })

if classification_outputs:
    classification_summary = pd.DataFrame(classification_outputs)
    display(classification_summary)
else:
    print("No slides were classified.")


In [ ]:
# Cell strategy:
# Optionally export each classified table to a fov/label-keyed CSV.
# This export keeps the classification and marker columns together and
# does not rely on per-cell x/y coordinates.

qupath_exports = []

if EXPORT_CLASSIFIED_CSV:
    for row in classification_outputs:
        csv_path = export_to_qupath(
            classified_csv=row["classified_csv"],
            class_cols=EXPORT_CLASS_COLUMNS,
        )
        qupath_exports.append({
            "slide": row["slide"],
            "qupath_csv": csv_path,
        })

if qupath_exports:
    display(pd.DataFrame(qupath_exports))
elif EXPORT_CLASSIFIED_CSV:
    print("No CSV exports were created.")
else:
    print("EXPORT_CLASSIFIED_CSV is False; skipping CSV export.")


## Notes

- Edit `MARKERS`, `COL_MAP`, `MUTEX_PAIRS`, and `CELL_TYPE_RULES` before trusting the outputs.
- If a cell matches multiple class rules, put more specific rules first. The first match becomes `Cell_Type`; later matches are stored in `Additional_Labels`.
- `EXPORT_CLASSIFIED_CSV` is optional and produces a `fov`/`label`-keyed CSV rather than a coordinate-based point table.
